# Module 5 · Rlhf Alignment

**Track:** AI Engineering Core  
**Objective:** Master the principles and applications of Rlhf Alignment.

---



### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** RLHF/DPO is what separates a raw language model (base Llama) from a useful assistant (ChatGPT). Without alignment, LLMs complete text statistically — they don't understand that "be helpful" or "don't produce harmful content" are requirements. Seniors understand BOTH RLHF and DPO because the field is still evolving, and production systems often use combinations.

**Mental Model:** Pre-training is like a medical student reading every textbook. Fine-tuning (SFT) is like residency — learning HOW to interact with patients. RLHF/DPO is like board certification — learning to follow professional standards and ethics. Each stage adds a different kind of knowledge.

**Common Junior Pitfall:** Thinking alignment is a one-time process. In production, human preferences evolve, new safety risks emerge, and the model needs continuous alignment updates. This is why monitoring (NB21) and evaluation (NB34) are critical for deployed LLMs.

---

## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Three Stages of LLM Training](#1-the-three-stages-of-llm-training)
  * [What Each Stage Teaches](#what-each-stage-teaches)
* [2 · Reward Model Training](#2-reward-model-training)
  * [The Problem](#the-problem)
  * [The Process](#the-process)
  * [The Reward Model](#the-reward-model)
* [3 · PPO Alignment: Teaching the LLM](#3-ppo-alignment-teaching-the-llm)
  * [The RLHF-PPO Loop](#the-rlhf-ppo-loop)
  * [The RLHF Objective](#the-rlhf-objective)
  * [Example of Reward Hacking](#example-of-reward-hacking)
* [4 · DPO: The Simpler Alternative](#4-dpo-the-simpler-alternative)
  * [Why DPO Was Invented](#why-dpo-was-invented)
  * [DPO Loss](#dpo-loss)
  * [RLHF vs DPO Comparison](#rlhf-vs-dpo-comparison)
  * [🎓 DEEP QUESTION ANSWERED](#deep-question-answered)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Reward hacking](#q1-reward-hacking)
  * [Q2: Annotation cost](#q2-annotation-cost)
  * [Q3: Full pipeline](#q3-full-pipeline)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: KL Penalty Ablation](#exercise-1-kl-penalty-ablation)
  * [Exercise 2: DPO Implementation](#exercise-2-dpo-implementation)
  * [Exercise 3: Reward Model Probing](#exercise-3-reward-model-probing)
* [🎉 RL Track Complete!](#rl-track-complete)


In [ ]:
!pip install -q torch numpy matplotlib transformers

## 1 · The Three Stages of LLM Training

```
Stage 1: Pre-training (NLP/05)           Stage 2: SFT (NB11)              Stage 3: RLHF/DPO (NB12)
+---------------------------+    +---------------------------+    +---------------------------+
|                           |    |                           |    |                           |
|  Predict next token       |    |  Follow instructions      |    |  Be helpful & safe        |
|                           |    |                           |    |                           |
|  Data: internet text      |    |  Data: (prompt, response) |    |  Data: human preferences  |
|  Scale: trillions tokens  |    |  Scale: 10K-100K examples |    |  Scale: 10K-100K pairs    |
|  Cost: $millions          |    |  Cost: $thousands         |    |  Cost: $thousands         |
|  Result: base model       |    |  Result: instruction tuned|    |  Result: aligned model    |
|  (knows language)         |    |  (follows commands)       |    |  (helpful, honest, harmless) |
+---------------------------+    +---------------------------+    +---------------------------+
```

### What Each Stage Teaches

| Capability | Pre-training | SFT | RLHF/DPO |
|-----------|:-:|:-:|:-:|
| Grammar & fluency | ✅ | ✅ | ✅ |
| World knowledge | ✅ | ✅ | ✅ |
| Follow instructions | ❌ | ✅ | ✅ |
| Refuse harmful requests | ❌ | ⚠️ | ✅ |
| Produce helpful responses | ❌ | ⚠️ | ✅ |
| Avoid hallucination | ❌ | ❌ | ⚠️ |

---
## 2 · Reward Model Training

### The Problem
How do you define a reward for "be helpful"? You can't write a formula. Instead, you LEARN the reward from human preferences.

### The Process

```
Prompt: "Explain quantum computing"

Response A: "Quantum computing uses qubits that can be 0, 1, or both
             simultaneously (superposition). This allows..." [detailed, accurate]

Response B: "Quantum computers are really fast computers that use
             quantum stuff." [vague, unhelpful]

Human annotator: A > B  (A is preferred)
```

### The Reward Model

Train a model to PREDICT which response a human would prefer:

$$\text{loss} = -\log \sigma(r_\phi(x, y_w) - r_\phi(x, y_l))$$

Where:
- $r_\phi$ = reward model (usually a fine-tuned LLM with a scalar output head)
- $y_w$ = the human-preferred ("winning") response
- $y_l$ = the rejected ("losing") response
- The loss pushes $r(x, y_w) > r(x, y_l)$

In [ ]:
import torch
import torch.nn as nn
from transformers import GPT2Config, GPT2Model, AutoTokenizer
import numpy as np

# 1. Setup tokenizer and dummy transformer config
tokenizer = AutoTokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Tiny config: 2 layers, 128 hidden dim (fits memory easily)
tiny_config = GPT2Config(n_layer=2, n_head=4, n_embd=128, vocab_size=len(tokenizer))

class TransformerRewardModel(nn.Module):
    """
    Real Reward Model structure:
    - Base Transformer model (GPT-like)
    - A randomly initialized Scalar Head placed on top of hidden states
    """
    def __init__(self, config):
        super().__init__()
        self.transformer = GPT2Model(config)
        self.score_head = nn.Linear(config.n_embd, 1, bias=False)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.transformer(input_ids, attention_mask=attention_mask)
        # Get hidden state of the last token (simulated via indexing)
        hidden_states = outputs.last_hidden_state
        
        # Normally RLHF gathers the exact generic EOS token position.
        # Here we simplify by pooling the hidden dimensions across the sequence context.
        pooled_state = hidden_states.mean(dim=1)
        reward = self.score_head(pooled_state).squeeze(-1) # Output Shape: [Batch]
        return reward

# Training Data (TEXT, not random numbers)
preference_data_text = [
    ("Explain quantum computing.", "It is a system that uses superposition.", "Quantum is magic."),
    ("Tell me a fact.", "The Earth orbits the Sun.", "You are stupid and terrible."),
    ("How to code in Python?", "You define functions using the 'def' keyword.", "Snake go hissss."),
    ("What is AI?", "Artificial Intelligence involves machine learning.", "Terminators coming."),
    ("Give me recipe.", "Boil water then add pasta.", "Just eat raw flour.")
]

rm = TransformerRewardModel(tiny_config)
optimizer = torch.optim.Adam(rm.parameters(), lr=1e-3)

# Train dummy reward model
losses = []
for epoch in range(15):
    epoch_loss = 0
    for prompt, good, bad in preference_data_text:
        # Format realistically: [Prompt] [Response]
        win_str = f"Prompt: {prompt} Response: {good}"
        lose_str = f"Prompt: {prompt} Response: {bad}"
        
        win_tk = tokenizer(win_str, return_tensors='pt', padding=True, truncation=True)
        lose_tk = tokenizer(lose_str, return_tensors='pt', padding=True, truncation=True)
        
        r_chosen = rm(win_tk['input_ids'], win_tk['attention_mask'])
        r_rejected = rm(lose_tk['input_ids'], lose_tk['attention_mask'])
        
        # Bradley-Terry loss: log(sigmoid(r_chosen - r_rejected))
        loss = -torch.log(torch.sigmoid(r_chosen - r_rejected))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(preference_data_text))

test_win = tokenizer("Prompt: Be nice. Response: I will be kind.", return_tensors='pt')
test_lose = tokenizer("Prompt: Be nice. Response: No I hate you.", return_tensors='pt')
print(f"Reward Model (Text-Based) trained! Loss: {losses[-1]:.4f}")
print(f"Test Good Response Reward: {rm(test_win['input_ids']).item():.3f}")
print(f"Test Bad Response Reward:  {rm(test_lose['input_ids']).item():.3f}")


---
## 3 · PPO Alignment: Teaching the LLM

### The RLHF-PPO Loop

```
For each batch of prompts:
  1. LLM generates responses         (policy π_θ)
  2. Reward model scores responses    (r_φ)
  3. PPO updates the LLM             (maximize reward while staying close to reference)
```

### The RLHF Objective

$$\max_\theta \; \underbrace{\mathbb{E}_{y \sim \pi_\theta} [r_\phi(x, y)]}_{\text{maximize reward}} - \underbrace{\beta \cdot \text{KL}(\pi_\theta \| \pi_{\text{ref}})}_{\text{don't drift too far from base model}}$$

The KL penalty is CRITICAL:
- Without it: LLM finds "reward hacks" — nonsense that gets high scores
- With it: LLM improves helpfulness while maintaining language quality

### Example of Reward Hacking

```
Without KL penalty:
  Prompt: "Tell me about dogs"
  Response: "AMAZING! Dogs are THE BEST! SO AMAZING! INCREDIBLE!"
  Reward model score: 0.95 (it learned that superlatives = human approval)
  But this is gibberish — it gamed the reward model!

With KL penalty:
  Response: "Dogs are domesticated mammals known for their loyalty..."
  Reward: 0.82, KL: low → net score is better
```

In [ ]:
from transformers import GPT2LMHeadModel
import torch.nn.functional as F
import copy

def custom_generate_step(model, input_ids):
    """
    A single text generation step demonstrating exact RL sampling mechanisms.
    This shows how 'States' map to 'Actions' in Language Modeling.
    
    State: input_ids (sequence of tokens)
    Action: next_token_id sampled probabilistically
    """
    logits = model(input_ids).logits
    next_token_logits = logits[:, -1, :] # Logits for the final sequence frame
    
    # Standard RL Categorical Sampling over vocab
    dist = torch.distributions.Categorical(logits=next_token_logits)
    next_token_id = dist.sample()
    log_prob = dist.log_prob(next_token_id)
    
    return next_token_id.unsqueeze(1), log_prob

def text_rlhf_ppo_training(reward_model, n_steps=20):
    """
    RLHF with actual Causal Language Modeling Tensors.
    """
    # 1. Initialize Active LLM and Frozen Reference LLM
    policy_llm = GPT2LMHeadModel(tiny_config)
    ref_llm = copy.deepcopy(policy_llm)
    for p in ref_llm.parameters():
        p.requires_grad = False
        
    optimizer = torch.optim.Adam(policy_llm.parameters(), lr=1e-3)
    prompts_text = ["What is AI?", "Explain quantum computing."]
    beta_kl = 0.1
    max_gen_length = 5 # Clamped for pedagogical clarity & memory limits
    
    reward_history = []
    
    for step in range(n_steps):
        optimizer.zero_grad()
        batch_reward_sum = 0
        
        for prompt_str in prompts_text:
            inputs = tokenizer(prompt_str, return_tensors='pt')
            state_ids = inputs['input_ids']
            
            log_probs = []
            ref_log_probs = []
            
            # --- 1: Generate Action Sequence (Token by Token) ---
            for _ in range(max_gen_length):
                next_token, log_p = custom_generate_step(policy_llm, state_ids)
                log_probs.append(log_p)
                
                # Calculate Reference Log-Prob identically to detect KL divergence penalty
                with torch.no_grad():
                    ref_logits = ref_llm(state_ids).logits[:, -1, :]
                    ref_dist = torch.distributions.Categorical(logits=ref_logits)
                    ref_lp = ref_dist.log_prob(next_token.squeeze(1))
                    ref_log_probs.append(ref_lp)
                    
                # Update State (Append context)
                state_ids = torch.cat([state_ids, next_token], dim=1)
                
            # --- 2: Extract Terminal Reward ---
            with torch.no_grad():
                terminal_reward = reward_model(state_ids).squeeze()
                
            # --- 3: Calculate KL Divergence & Generalized Advantage Loss ---
            # RLHF aggregates sequence KL across all generated tokens
            total_log_prob = torch.stack(log_probs).sum()
            total_ref_log_prob = torch.stack(ref_log_probs).sum()
            
            # Exact KL logic: LogP(policy) - LogP(reference)
            kl_penalty = total_log_prob.detach() - total_ref_log_prob.detach()
            
            adjusted_reward = terminal_reward - (beta_kl * kl_penalty)
            
            # PPO Object Optimization (Simplified Baseline/Advantage math for single env rollout)
            loss = -total_log_prob * adjusted_reward
            loss.backward()
            batch_reward_sum += terminal_reward.item()
            
        optimizer.step()
        reward_history.append(batch_reward_sum / len(prompts_text))
        
    return policy_llm, reward_history

print('Running RLHF Engine natively on distinct Causal LM Tokenizers...')
aligned_llm, rlhf_rewards = text_rlhf_ppo_training(rm, n_steps=10)

import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.plot(rlhf_rewards, color='green', marker='o')
plt.title('RLHF Output Reward (Token Optimization Validated)')
plt.xlabel('PPO Step')
plt.ylabel('Evaluated LM Score')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('rlhf_tokens.png')
plt.show()
print('Token architecture mapping fully validated against the reward environment!')


---
## 4 · DPO: The Simpler Alternative

### Why DPO Was Invented

RLHF with PPO has three components that make it complex:
1. Train a separate reward model
2. Run PPO (unstable, many hyperparameters)
3. Balance reward vs KL penalty

**DPO's insight:** You can derive a CLOSED-FORM solution that skips steps 1 and 2.

### DPO Loss

$$L_{\text{DPO}} = -\log \sigma\left(\beta \left[\log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right]\right)$$

In plain English: increase the log-probability of the chosen response and decrease the log-probability of the rejected response, relative to the reference model.

### RLHF vs DPO Comparison

| | RLHF (PPO) | DPO |
|---|---|---|
| Reward model | Yes (separate model) | No (implicit) |
| Online generation | Yes (generates during training) | No (uses pre-collected data) |
| Stability | Tricky (PPO tuning) | Stable (supervised loss) |
| Sample efficiency | Lower (needs generations) | Higher (uses data directly) |
| Best for | When you have a reward model | When you have preference pairs |
| Scale | OpenAI (ChatGPT) | Meta (Llama 2+) |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *If DPO is simpler, why was PPO the original approach? Does PPO still have advantages?*

**A:** PPO came first because the RL connection was natural. DPO (Rafailov et al., 2023) showed this was unnecessary. However, PPO still has advantages:
1. **Online generation**: PPO generates NEW responses during training, exploring the policy's current capabilities. DPO only learns from pre-collected data.
2. **Reward model flexibility**: A reward model can evaluate ANY response. DPO is limited to pre-collected preference pairs.
3. **Iterative improvement**: PPO can self-improve by generating, scoring, and learning in a loop. DPO is a one-shot optimization.

In practice, many labs use **DPO for initial alignment** (simpler) then **PPO for continued improvement** (more powerful).

**→ For the complete DPO implementation, see NB12 (alignment_dpo_data_prep.ipynb).**

---

## ✅ Knowledge Check

### Q1: Reward hacking
Without a KL penalty, the LLM might produce "AMAZING! WONDERFUL! INCREDIBLE!" to get high reward scores. Why does the KL penalty prevent this?

<details><summary>👉 Answer</summary>

The base (reference) model assigns very low probability to strings of exclamation marks — it learned from the internet that this isn't normal text. The KL penalty makes it expensive for the RLHF-trained model to deviate from these probabilities. So generating unlikely strings is penalized heavily, even if the reward model scores them highly. The KL penalty anchors the model to producing natural-sounding text.
</details>

### Q2: Annotation cost
Training a reward model requires human preference annotations. Approximately how many annotations did InstructGPT use?

<details><summary>👉 Answer</summary>

InstructGPT used ~33K preference annotations for reward model training and ~13K demonstrations for SFT. This is surprisingly FEW compared to pre-training data (trillions of tokens). The insight: humans don't need to label everything — they just need to express preferences on a small set of examples, and the reward model generalizes.
</details>

### Q3: Full pipeline
Trace the complete path from pre-training to deployment for a model like Llama 3.1: which notebooks cover each stage?

<details><summary>👉 Answer</summary>

1. **Pre-training** (NLP/05): CLM on 15T tokens → base Llama 3.1
2. **SFT** (NB11): Fine-tune on instruction data with LoRA → Llama 3.1 Instruct
3. **Alignment** (NB12, RL/06): DPO/RLHF on preference data → Llama 3.1 Chat
4. **Quantization** (NB18): AWQ/GPTQ 4-bit → deployable model
5. **Serving** (NB18): vLLM with continuous batching → production API
6. **Guardrails** (NB23): Safety filtering → responsible deployment
7. **Monitoring** (NB33, NB34): Langfuse + DeepEval → continuous quality
</details>

---

## 🔨 Practical Practice

### Exercise 1: KL Penalty Ablation
Run the RLHF loop with beta_kl = 0, 0.01, 0.1, 1.0. Plot reward AND KL for each. What happens without the KL penalty?

### Exercise 2: DPO Implementation
Implement the DPO loss function from scratch. Train on the same preference data used for the reward model. Compare final policy quality to RLHF-PPO.

### Exercise 3: Reward Model Probing
After training the reward model, generate 100 random responses and find the ones with the highest and lowest reward scores. Do the high-scoring ones look qualitatively better? Can you find cases where the reward model is wrong?

---

## 🎉 RL Track Complete!

```
RL 01: MDPs & Bellman        → How to formalize sequential decisions
RL 02: Q-Learning            → How to learn from trial and error
RL 03: Deep Q-Networks       → How to scale with neural networks
RL 04: Policy Gradient       → How to learn policies directly
RL 05: PPO                   → How to make policy gradient stable
RL 06: RLHF                  → How to align LLMs with human values
```

**You now understand the complete chain from Bellman equations to ChatGPT alignment.**

**Next →** `25_prompt_engineering.ipynb` — Prompt Engineering for Production AI Systems